## Tutorial 3 - Preparing for production

To deal with any steric clashes that could have been present in the input system or introduced in the coarse-graining stage or in bilayer building, we need to energy-minimise the system. For coarse-grained systems, this will normally suffice to be able to seed a production run, but care should be taken to disregard the start of the production simulation as the system equilibrates.

The most widely used simulation engine for Martini simulations is GROMACS, which we will use here. There is now an implementation of [Martini in OpenMM](https://www.cell.com/biophysj/fulltext/S0006-3495(23)00237-0?_returnURL=https%3A%2F%2Flinkinghub.elsevier.com%2Fretrieve%2Fpii%2FS0006349523002370%3Fshowall%3Dtrue) which can also be used.   

The `grompp` step in GROMACS is used to prepare a production file, which is an output in the form of a `.tpr` file. The inputs are:
- `-c` a coordinate file (`.pdb` or `.gro`)
- `-p` the topology file, showing what is contained within the system and information about the forcefield/parameter files (`.top` file)
- `-f` an `.mdp` file, which contains the settings for this simulation run
- `-maxwarn` which suppresses warnings. **This should not be used unless you know what you are doing**, but used below to dismiss warnings about atom name changes (which happened with new lipid parameters) and another warning that we can disregard for now

We can have a look at what is contained within this energy minimisation file:

In [ ]:
!head mdps/em.mdp

- `integrator` defines what type of simulation is performed; in this case it uses the steepest decents algorithm to perform energy minimisation  
- `nsteps` defines the maximum number of steps performed  
- `emtol` is the cutoff where we define the system as energy minimised, a target energy (in kJ mol<sup>-1</sup>)  
- `emstep` is the largest displacement allowed per step performed (in nm)  

We can now use this file and the files we created in the previous step to generate a `.tpr` file

In [ ]:
%%bash 

cp ../tutorial_2/system.gro ../tutorial_2/topol.top .

cp ../tutorial_2/*.itp  itps/

gmx grompp -f mdps/em.mdp -c system.gro -p topol.top -o em.tpr -maxwarn 2

We can now run our energy minimisation. We do this with the `gmx mdrun` command, which takes the following inputs:

- `-deffnm` sets the default file names and uses this to find the .tpr file
- `-v` to be verbose and print all the steps it is taking (and when it might finish)
- `-ntmpi` number of thread-MPI ranks to start

In [ ]:
%%bash

gmx mdrun -deffnm em -v -ntmpi 1

We now have an energy minimised system! We can have a look at this and hopefully you can see the difference:

In [ ]:
from IPython.display import HTML, display
import base64, os, json


# ══════════════════════════════════════════════════════════════════════════════
# ── Built-in residue-name sets
# ══════════════════════════════════════════════════════════════════════════════

_PROTEIN_RESNAMES: set = {
    # Standard amino acids (GROMACS / AMBER / CHARMM conventions)
    'ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY',
    'HIS', 'HIE', 'HID', 'HIP', 'HSE', 'HSD', 'HSP',
    'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR',
    'TRP', 'TYR', 'VAL',
    # Terminal caps
    'ACE', 'NME', 'NHE', 'NH2', 'CT3',
}

_WATER_RESNAMES: set = {
    'W'
}


# ══════════════════════════════════════════════════════════════════════════════
# ── Structure-splitting helpers
# ══════════════════════════════════════════════════════════════════════════════

def _split_by_resnames(text: str, fmt: str, resnames: set) -> tuple:
    """
    Partition a PDB or GRO structure string into (matched_text, unmatched_text).

    Atom records whose residue name is in resnames go to matched; all others
    go to unmatched.  Header / footer records (REMARK, CONECT, END, GRO box
    vector) are duplicated into both outputs so each sub-file is self-consistent
    and parseable by Mol*.

    Parameters
    ----------
    text     : str   Full file contents decoded as a string.
    fmt      : str   'pdb' or 'gro'.
    resnames : set   Residue names to match (case-sensitive).

    Returns
    -------
    (matched_text, unmatched_text) : (str, str)
        matched_text is '' when resnames is empty or fmt is unsupported.
    """
    if not resnames:
        return '', text

    if fmt == 'pdb':
        matched, unmatched = [], []
        for line in text.splitlines(keepends=True):
            if line[:6] in ('ATOM  ', 'HETATM'):
                # Residue name: columns 17-20 (4-char handles non-standard names)
                (matched if line[17:21].strip() in resnames else unmatched).append(line)
            else:
                # Headers, REMARK, CONECT, END → keep in both
                matched.append(line)
                unmatched.append(line)
        return ''.join(matched), ''.join(unmatched)

    elif fmt == 'gro':
        lines = text.splitlines(keepends=True)
        if len(lines) < 3:
            return '', text
        title = lines[0]
        try:
            n = int(lines[1].strip())
        except ValueError:
            return '', text
        box = lines[2 + n] if len(lines) > 2 + n else '\n'

        matched, unmatched = [], []
        for line in lines[2:2 + n]:
            # GRO residue name: columns 5-9 (0-indexed)
            rn = line[5:10].strip() if len(line) >= 10 else ''
            (matched if rn in resnames else unmatched).append(line)

        m_text = title + f'{len(matched)}\n'   + ''.join(matched)   + box
        u_text = title + f'{len(unmatched)}\n' + ''.join(unmatched) + box
        return m_text, u_text

    else:
        print(
            f"[view_martini] Warning: component splitting requires PDB or GRO "
            f"(got '{fmt}').  Loading full structure as a single layer."
        )
        return text, ''


def _has_atoms(text: str, fmt: str) -> bool:
    """Return True if the structure text contains at least one atom record."""
    if fmt == 'pdb':
        return any(ln[:6] in ('ATOM  ', 'HETATM') for ln in text.splitlines())
    if fmt == 'gro':
        lines = text.splitlines()
        try:
            return len(lines) >= 2 and int(lines[1].strip()) > 0
        except ValueError:
            return False
    return bool(text.strip())


def _color_theme(color: str) -> dict:
    """
    Convert a color string to a Mol* colorTheme descriptor dict.

    A CSS hex string (e.g. '#00BFFF') → uniform theme with the colour
    encoded as a 24-bit integer (Mol*'s internal Color type).
    Any other string → Mol* named theme (e.g. 'chain-id', 'residue-name').
    """
    if isinstance(color, str) and color.startswith('#'):
        return {'name': 'uniform', 'params': {'value': int(color.lstrip('#'), 16)}}
    return {'name': color}


# ══════════════════════════════════════════════════════════════════════════════
# ── HTML / iframe builder
# ══════════════════════════════════════════════════════════════════════════════

def _build_html(components: list, height: int) -> HTML:
    """
    Assemble the Mol* iframe page from a list of component descriptor dicts.

    Each dict must contain:
        b64           : str   base64-encoded structure text
        fmt           : str   Mol* format string ('pdb', 'gro', 'mmcif')
        label         : str   display label (shown in Mol* state tree)
        spacefillSize : float bead sphere radius in Å
        alpha         : float sphere opacity (0–1)
        showBonds     : bool  whether to render bond sticks
        stickRadius   : float bond stick radius in Å
        colorTheme    : dict  Mol* colorTheme descriptor
    """
    comps_json = json.dumps(components)

    # Note: Python f-string — all JS curly braces are doubled {{ }}
    #       except Python variables being interpolated: {comps_json}, {height}
    page = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8"/>
  <link rel="stylesheet"
        href="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.css"/>
  <style>
    html, body {{ margin:0; padding:0; height:100%; background:#1a1a2e; }}
    #app {{ position:absolute; inset:0; }}
  </style>
</head>
<body>
  <div id="app"></div>
  <script src="https://cdn.jsdelivr.net/npm/molstar@latest/build/viewer/molstar.js"></script>
  <script>
  (async () => {{

    // ── 1. Create Mol* viewer ────────────────────────────────────────────────
    const viewer = await molstar.Viewer.create('app', {{
      layoutIsExpanded:      false,
      layoutShowControls:    false,
      layoutShowLeftPanel:   false,
      layoutShowSequence:    false,
      layoutShowLog:         false,
      viewportShowAnimation: false,
      viewportShowExpand:    true,
    }});
    const plugin = viewer.plugin;

    // ── 2. Component descriptors injected from Python ────────────────────────
    const components = {comps_json};

    // ── 3. Helper: load one component and apply its representations ──────────
    async function loadComponent(comp) {{
      const rawData = await plugin.builders.data.rawData(
        {{ data: atob(comp.b64) }},
        {{ state: {{ isGhost: true }} }}      // hide raw-data node from the UI tree
      );
      const traj   = await plugin.builders.structure.parseTrajectory(rawData, comp.fmt);
      const model  = await plugin.builders.structure.createModel(traj);
      const struct = await plugin.builders.structure.createStructure(model,
                       {{ label: comp.label }});

      // VdW spacefill — one sphere per CG bead
      await plugin.builders.structure.representation.addRepresentation(struct, {{
        type:       'spacefill',
        colorTheme: comp.colorTheme,
        sizeTheme:  {{ name: 'uniform', params: {{ value: comp.spacefillSize }} }},
        typeParams: {{ alpha: comp.alpha }},
      }});

      // Bond sticks (optional per component)
      if (comp.showBonds) {{
        await plugin.builders.structure.representation.addRepresentation(struct, {{
          type:       'ball-and-stick',
          colorTheme: comp.colorTheme,
          sizeTheme:  {{ name: 'uniform' }},
          typeParams: {{
            sizeFactor:     comp.stickRadius,
            ballSizeFactor: 0.01,   // near-zero ball — spheres handled by spacefill
            bondSpacing:    1.0,
          }},
        }});
      }}
    }}

    // ── 4. Load every component then reset camera ────────────────────────────
    for (const comp of components) {{
      await loadComponent(comp);
    }}
    plugin.canvas3d?.requestCameraReset();

  }})();
  </script>
</body>
</html>"""

    escaped = page.replace("'", "&#39;")
    return HTML(
        f'<iframe srcdoc=\'{escaped}\' '
        f'style="width:100%;height:{height}px;border:none;border-radius:6px;"></iframe>'
    )


# ══════════════════════════════════════════════════════════════════════════════
# ── Main viewer function
# ══════════════════════════════════════════════════════════════════════════════

def view_martini(
    structure_file: str,
    height: int = 550,

    # ── Global defaults (per-component values fall back to these) ─────────────
    bead_radius: float = 2.6,       # Å — Regular (R) bead; Tiny≈2.1, Small≈2.3
    sphere_alpha: float = 0.45,     # 0=transparent, 1=fully opaque
    stick_radius: float = 0.15,     # Å — bond stick radius

    # ── Protein ───────────────────────────────────────────────────────────────
    show_protein: bool = True,
    protein_names: list = None,     # None → built-in amino-acid residue list
    protein_color: str = "chain-id",
    protein_bead_radius: float = None,
    protein_alpha: float = None,
    show_protein_bonds: bool = True,

    # ── Lipids ────────────────────────────────────────────────────────────────
    show_lipids: bool = True,       # only renders if lipid_names is also set
    lipid_names: list = None,       # e.g. ['POPC', 'POPE', 'CHOL']
    lipid_color: str = "residue-name",
    lipid_bead_radius: float = None,
    lipid_alpha: float = None,
    show_lipid_bonds: bool = True,

    # ── Water ─────────────────────────────────────────────────────────────────
    show_water: bool = False,       # off by default — can be millions of beads!
    water_names: list = None,       # None → ['W', 'WF']
    water_color: str = "#00BFFF",   # deep sky blue
    water_bead_radius: float = 1.9, # Martini 3 W bead
    water_alpha: float = 0.25,      # very transparent — don't obscure protein

    # ── Ions ──────────────────────────────────────────────────────────────────
    show_ions: bool = True,         # only renders if ion_names is also set
    ion_names: list = None,         # e.g. ['NA', 'CL', 'CA']
    ion_color: str = "#FFD700",     # gold
    ion_bead_radius: float = 1.5,   # Å
    ion_alpha: float = 0.9,

) -> HTML:
    """
    Visualise a Martini 3 coarse-grained structure in Mol* inside Jupyter.

    The structure is partitioned in Python into up to four independent layers
    (protein / lipids / water / ions).  Each layer is loaded as a separate
    Mol* structure so it can have its own bead size, colour, opacity, and
    bond visibility — without interfering with the other layers.

    Parameters
    ----------
    structure_file : str
        Path to a PDB or GRO file.  mmCIF is accepted but component splitting
        is not supported; the full structure is loaded as a single layer.

    height : int
        Viewer height in pixels (default 550).

    bead_radius : float
        Default spacefill sphere radius in Å.  Martini 3 bead sizes:
            Regular (R) ≈ 2.6 Å  ← default
            Small   (S) ≈ 2.3 Å
            Tiny    (T) ≈ 2.1 Å
        Override per-component with protein_bead_radius, lipid_bead_radius, etc.

    sphere_alpha : float
        Default sphere opacity (0–1, default 0.45).  Semi-transparent lets
        bond sticks show through the beads.

    stick_radius : float
        Bond stick radius in Å (default 0.15).

    show_protein : bool
        Render protein beads (default True).

    protein_names : list of str, optional
        Residue names to classify as protein.  ``None`` uses the built-in set
        of standard 3-letter amino-acid codes plus common terminal caps.

    protein_color : str
        Mol* colour theme name or CSS hex colour string.
        Named themes: 'chain-id' (default), 'residue-name', 'element-symbol',
                      'secondary-structure', 'uncertainty', 'sequence-id'.
        Hex colours:  '#FF6600', '#00CCFF', etc.

    protein_bead_radius : float, optional
        Override bead_radius for protein beads only.

    protein_alpha : float, optional
        Override sphere_alpha for protein beads only.

    show_protein_bonds : bool
        Render bond sticks between protein beads (default True).

    show_lipids : bool
        Render lipid beads (default True).  Has no effect unless
        lipid_names is also provided.

    lipid_names : list of str, optional
        Residue names to classify as lipids.  Common Martini 3 examples:
            PC lipids : 'POPC', 'DOPC', 'DPPC', 'DLPC'
            PE lipids : 'POPE', 'DOPE'
            Sterols   : 'CHOL', 'ERG'
            Others    : 'POPS', 'POPA', 'POPG', 'PIP2', 'CARD', ...

    lipid_color : str
        Colour theme for lipids.  Default ``'residue-name'`` assigns each
        lipid type its own colour — useful for mixed membranes.

    lipid_bead_radius : float, optional
        Override bead_radius for lipid beads only.

    lipid_alpha : float, optional
        Override sphere_alpha for lipid beads only.

    show_lipid_bonds : bool
        Render bond sticks between lipid beads (default True).

    show_water : bool
        Render water beads (default **False**).  A typical simulation box
        contains enormous numbers of W beads; enable only for small test
        systems to avoid freezing the browser.

    water_names : list of str, optional
        Residue names to classify as water.  Defaults to ``['W', 'WF']``
        (standard + antifreeze Martini 3 water).

    water_color : str
        Colour for water spheres (default ``'#00BFFF'`` deep sky blue).

    water_bead_radius : float
        Spacefill radius for water beads in Å (default 1.9).

    water_alpha : float
        Opacity for water spheres (default 0.25 — very transparent).

    show_ions : bool
        Render ion beads (default True).  Has no effect unless
        ion_names is also provided.

    ion_names : list of str, optional
        Residue names to classify as ions.  Common Martini 3 examples:
            'NA'  (sodium),  'CL'  (chloride),  'CA'  (calcium),
            'MG'  (magnesium), 'K'  (potassium), 'ZN'  (zinc).
        Check your ITP / force-field docs for the exact names used.

    ion_color : str
        Colour for ion spheres (default ``'#FFD700'`` gold).

    ion_bead_radius : float
        Spacefill radius for ion beads in Å (default 1.5).

    ion_alpha : float
        Opacity for ion spheres (default 0.9).

    Returns
    -------
    IPython.display.HTML
        Call display() on the result, or let Jupyter auto-display it.

    Notes
    -----
    * Bond connectivity is inferred from bead-bead distances by Mol*.
      For exact Martini connectivity supply a PDB with CONECT records
      (generated from your ITP [ bonds ] section).
    * Residues not matching any of the four categories are silently omitted.
      Use protein_names / lipid_names / ion_names to capture anything unusual.
    * Each layer is base64-encoded and embedded in the HTML — no HTTP server
      is required; the notebook cell is fully self-contained.
    """

    # ── 1. Load and decode the structure file ─────────────────────────────────
    with open(structure_file, 'rb') as fh:
        raw_bytes = fh.read()
    raw_text = raw_bytes.decode('utf-8', errors='replace')

    ext = os.path.splitext(structure_file)[1].lstrip('.').lower()
    fmt = {'gro': 'gro', 'pdb': 'pdb', 'cif': 'mmcif', 'mmcif': 'mmcif'}.get(ext, 'pdb')

    # mmCIF: no component splitting — load whole file as a single layer
    if fmt == 'mmcif':
        print(
            "[view_martini] Note: mmCIF detected — component splitting is not "
            "supported.  Loading the full structure as a single layer."
        )
        comps = [{
            'b64':           base64.b64encode(raw_bytes).decode(),
            'fmt':           'mmcif',
            'label':         'Structure',
            'spacefillSize': protein_bead_radius or bead_radius,
            'alpha':         protein_alpha or sphere_alpha,
            'showBonds':     show_protein_bonds,
            'stickRadius':   stick_radius,
            'colorTheme':    _color_theme(protein_color),
        }]
        return _build_html(comps, height)

    # ── 2. Resolve residue-name sets ──────────────────────────────────────────
    p_names = set(protein_names) if protein_names is not None else _PROTEIN_RESNAMES
    w_names = set(water_names)   if water_names  is not None else _WATER_RESNAMES
    l_names = set(lipid_names)   if lipid_names  is not None else set()
    i_names = set(ion_names)     if ion_names    is not None else set()

    # ── 3. Sequential peeling: each bead assigned to exactly one category ─────
    #
    #   remaining → strip protein → (protein_txt, remaining)
    #   remaining → strip lipids  → (lipid_txt,   remaining)
    #   remaining → strip water   → (water_txt,   remaining)
    #   remaining → strip ions    → (ion_txt,     _)
    #   (anything left is unclassified and not rendered)
    #
    remaining = raw_text

    protein_txt, remaining = _split_by_resnames(
        remaining, fmt, p_names if show_protein else set()
    )
    lipid_txt, remaining = _split_by_resnames(
        remaining, fmt, l_names if (show_lipids and l_names) else set()
    )
    water_txt, remaining = _split_by_resnames(
        remaining, fmt, w_names if show_water else set()
    )
    ion_txt, _ = _split_by_resnames(
        remaining, fmt, i_names if (show_ions and i_names) else set()
    )

    # ── 4. Build component descriptor list ────────────────────────────────────
    def make_comp(show, text, label, radius, alpha, color, bonds, bond_r=None):
        """Return a component dict if show=True and text contains atoms."""
        if not show or not _has_atoms(text, fmt):
            return None
        return {
            'b64':           base64.b64encode(text.encode('utf-8')).decode(),
            'fmt':           fmt,
            'label':         label,
            'spacefillSize': radius,
            'alpha':         alpha,
            'showBonds':     bonds,
            'stickRadius':   bond_r if bond_r is not None else stick_radius,
            'colorTheme':    _color_theme(color),
        }

    comps = list(filter(None, [
        make_comp(show_protein,
                  protein_txt, 'Protein',
                  protein_bead_radius or bead_radius,
                  protein_alpha       or sphere_alpha,
                  protein_color, show_protein_bonds),

        make_comp(bool(show_lipids and l_names),
                  lipid_txt, 'Lipids',
                  lipid_bead_radius or bead_radius,
                  lipid_alpha       or sphere_alpha,
                  lipid_color, show_lipid_bonds),

        make_comp(show_water,
                  water_txt, 'Water',
                  water_bead_radius, water_alpha,
                  water_color, False),            # no bonds for water

        make_comp(bool(show_ions and i_names),
                  ion_txt, 'Ions',
                  ion_bead_radius, ion_alpha,
                  ion_color, False),              # no bonds for ions
    ]))

    return _build_html(comps, height)

## Only line that you would need to change. Comment out each line in turn to strip the water/ions away to get a closer look at the membrane

# display(view_martini("em.gro",lipid_names=['POPC', 'POPE', 'POPI'], show_water=True, water_alpha=0.3, water_bead_radius=1.5, ion_names=['NA', 'CL']))

display(view_martini("em.gro",lipid_names=['POPC', 'POPE', 'POPI']))



Hopefully you can see that the system is less grid like and more like a system we would expect.

We can now use this to generate our production simulation. The `.mdp` file listing the settings is a lot longer and more complicated, we can see this below. 

In [ ]:
%%bash

head -n 44 mdps/5us-martini.mdp

We can highlight the most important settings here

- `integrator` this time is md, which will use an algorithm for integrating Newton's equation of motion (in this case using a leap-frog algorithm).
- `dt` which is the time step used by the integrator. For atomistic simulations, this is often 2 fs, but with Martini this can be extended to 20 fs.
- `nsteps` which, with the md integrator, is the number of steps that will happen. As we are now using a time-based integrator, we can calculate that 250000000 x 2 fs = 5 $\mu$s.
- `nstxout-compressed` is how many steps between saving coordinates into the `.xtc` format. As it stands, it is saved every nanosecond.
- `Pcoupltype` is the type of isotropy used for the pressure coupling. For soluble simulations, this will usually be set to isotropic, where each dimension (x,y and z) would be treated uniquely. When there is a membrane present, the x and y dimensions are intrinsically coupled, so we need to use the semiisotropic pressure coupling setting.
- `Pcouple` is the pressure coupling type to use, in this case the C-rescale algorithm.
- `tcoupl` is the temperature coupling type, in this case the v-rescale algorithm.
- `tc-groups` specifies groups to separate temperature baths. Separating into similarly mobile parts can help prevent excessive energies accumulating in one component.

To be able to determine the groups used in `tc-groups`, we need to make an index file. We can do that using a GROMACS command:

In [ ]:
%%bash

gmx make_ndx -f em.gro -o sys.ndx << EOF
rPOPC|rPOPE|rPOPI
name 18 LIPID
rW|rION
name 19 SOL_ION

q
EOF

Since the Protein is already listed in the index file, we do not need to specify this further. We need to group the lipids (**r**esidue POPE |-or etc) together and give this the appropriate name; same with the lipids and ions.

We are now ready to simulate our system! Depending on what you are studying, the number of repeats and length of simulation could differ. For investigating something such as lipid-protein interaction, I would usually start with 5 x 5 $\mu$s simulations, especially for a system of this size. We can set up one simulation below:

In [ ]:
%%bash

gmx grompp -f mdps/5us-martini.mdp -c em.gro -p topol.top -n sys.ndx -o md.tpr

Because of time constraints, here is a simulation I prepared earlier so we can move on to analysis of coarse-grained simulations